In [1]:
from mypackage.utilities import connect_to_db
from mypackage.mapping import reverse_combined_column_mapping
from mypackage.config import get_date_range,get_filename,bus_lines,get_foldername,groups_frontend,get_test_date,groups_middle
import pandas as pd
import numpy as np
import os

# 设定日期范围
date_range=get_date_range()

folder_name=get_foldername()
folder_path=os.path.join('Z:\\', '11-业务报表', '8.业务线核算', '其他','非业务线损益',folder_name)
# 路径不存在就新建一个
if not os.path.exists(folder_path):
    os.makedirs(folder_path,exist_ok=True)
    print(f"Created directory: {folder_path}")
    
# 获取全部部门中非业务线的收入明细
conn,cur=connect_to_db()
cur.execute(
  f"SELECT * FROM dim_org_struc WHERE bus_line='无' AND unique_lvl not like '%无归属%'"
)
df=cur.fetchall()
columns = [reverse_combined_column_mapping.get(desc[0]) for desc in cur.description]
df=pd.DataFrame(df,columns=columns)
# 不包含合伙人营销中心和审核能力中心
df=df[~df['分组简称'].isin(groups_frontend)]
levels=df['唯一层级'].drop_duplicates().tolist()

cur.execute(
  f"SELECT * FROM fact_revenue"
)
df=cur.fetchall()
columns = [reverse_combined_column_mapping.get(desc[0]) for desc in cur.description]
df_revenue=pd.DataFrame(df,columns=columns)
df_revenue=df_revenue[df_revenue['唯一层级'].isin(levels)]
df_revenue=df_revenue[df_revenue['会计期间'].isin(date_range)]
df_revenue=df_revenue.drop(['一级组织','二级组织','三级组织'],axis=1)
df_revenue[['一级组织','二级组织','三级组织']]=df_revenue['唯一层级'].str.split('-',n=2,expand=True)
df_revenue=df_revenue[['来源编号','一级组织','二级组织','三级组织','会计期间','财报合并','收入大类','产品大类','物料名称','不含税金额本位币','成本金额','运费成本','关税成本','软件成本']]
columns = bus_lines
df_revenue[columns] = np.nan
df_revenue.to_excel(f'{folder_path}//{get_filename("业务线收入拆分表")}.xlsx',index=False,sheet_name='部门金额')


In [2]:
from mypackage.utilities import connect_to_db
from mypackage.mapping import reverse_combined_column_mapping
import pandas as pd
import os
# 获取前中台部门中无法归属业务线的费用数据
conn,cur=connect_to_db()
cur.execute(
  f"SELECT * FROM dim_org_struc WHERE bus_line='无' AND (category='前台' or category='中台') AND unique_lvl not like '%无归属%'"
)
df=cur.fetchall()

columns = [reverse_combined_column_mapping.get(desc[0]) for desc in cur.description]
df=pd.DataFrame(df,columns=columns)
# 不包含合伙人营销中心和审核能力中心
df=df[(~df['分组简称'].isin(groups_frontend))&(~df['分组简称'].isin(groups_middle))]
levels=df['唯一层级'].drop_duplicates().tolist()
# 获取非业务线的费用明细
cur.execute(
  f"SELECT * FROM fact_expense"
)
df=cur.fetchall()
columns = [reverse_combined_column_mapping.get(desc[0]) for desc in cur.description]
df_expense=pd.DataFrame(df,columns=columns)
df_expense=df_expense[df_expense['唯一层级'].isin(levels)]
df_expense=df_expense[df_expense['会计期间'].isin(date_range)]
df_expense=df_expense.drop(['一级组织','二级组织','三级组织'],axis=1)
df_expense[['一级组织','二级组织','三级组织']]=df_expense['唯一层级'].str.split('-',n=2,expand=True)
df_expense=df_expense[['来源编号','一级组织','二级组织','三级组织','会计期间','财报合并','费用大类','摘要','费用金额','分摊业务线']]
columns = bus_lines
df_expense[columns] = np.nan
# 去掉零值
# 步骤1：先移除空值
df_expense = df_expense.dropna(subset=['费用金额'])
# 步骤2：再过滤掉0值
df_expense = df_expense[df_expense['费用金额'] != 0]
df_expense.to_excel(f'{folder_path}//{get_filename("业务线费用拆分表")}.xlsx',index=False,sheet_name='部门金额')

In [3]:
from mypackage.utilities import connect_to_db
from mypackage.mapping import reverse_combined_column_mapping
import pandas as pd
import numpy as np
import os
# 获取其他利润表项目无法归属业务线的明细
conn,cur=connect_to_db()
cur.execute(
  f"SELECT * FROM dim_org_struc WHERE bus_line='无' AND unique_lvl not like '%无归属%'"
)
df=cur.fetchall()
columns = [reverse_combined_column_mapping.get(desc[0]) for desc in cur.description]
df=pd.DataFrame(df,columns=columns)
# 不包含合伙人营销中心和审核能力中心
df=df[~df['分组简称'].isin(groups_frontend)]
levels=df['唯一层级'].drop_duplicates().tolist()
# 获取非业务线的收入明细
cur.execute(
  f"SELECT * FROM fact_profit_bd"
)
df=cur.fetchall()
acct_list=["信用减值损失", "公允价值变动收益" , "其他收益" , "所得税费用" , "投资收益" , "税金及附加" , "营业外支出" , "营业外收入" , "资产减值损失" , "资产处置收益"]
columns = [reverse_combined_column_mapping.get(desc[0]) for desc in cur.description]
df_profit=pd.DataFrame(df,columns=columns)
df_profit=df_profit[df_profit['唯一层级'].isin(levels) & df_profit['一级科目'].isin(acct_list)]
df_profit=df_profit[df_profit['日期'].isin(date_range)]
df_profit=df_profit.drop(['一级组织','二级组织','三级组织'],axis=1)
df_profit[['一级组织','二级组织','三级组织']]=df_profit['唯一层级'].str.split('-',n=2,expand=True)
df_profit=df_profit[['来源编号','一级组织','二级组织','三级组织','日期','财报合并','一级科目','本月金额']]
columns = bus_lines
df_profit[columns] = np.nan
# 去掉零值
# 步骤1：先移除空值
df_profit = df_profit.dropna(subset=['本月金额'])
# 步骤2：再过滤掉0值
df_profit = df_profit[df_profit['本月金额'] != 0]
df_profit.to_excel(f'{folder_path}//{get_filename("业务线其他拆分表")}.xlsx',index=False,sheet_name='部门金额')